In [0]:
# %sql
# WITH PRODUCT_KPIS AS (
#   SELECT
#     A.PRODUCT_ID,
#     A.PRODUCT_NAME,
#     A.CATEGORY,
#     SUM(B.QUANTITY) AS UNITS_SOLD,
#     COUNT(DISTINCT B.ORDER_ID) AS TOTAL_ORDERS,
#     SUM(B.GROSS_AMOUNT) AS GROSS_REVENUE,
#     SUM(B.DISCOUNT_AMOUNT) AS DISCOUNT_AMOUNT,
#     SUM(B.NET_AMOUNT) AS NET_REVENUE,
#     SUM(B.NET_AMOUNT) / NULLIF(SUM(B.quantity), 0) AS AVERAGE_SELLING_PRICE
#   FROM
#     ECOMMERCE.SILVER.PRODUCTS A
#       INNER JOIN ECOMMERCE.SILVER.ORDER_ITEMS B
#         ON A.PRODUCT_ID = B.PRODUCT_ID
#   GROUP BY
#     1,
#     2,
#     3
# ),
# CUSTOMER_KPIS AS (
#   SELECT
#     A.PRODUCT_ID,
#     COUNT(DISTINCT B.CUSTOMER_ID) AS UNIQUE_CUSTOMERS
#   FROM
#     ECOMMERCE.SILVER.ORDER_ITEMS A
#       INNER JOIN ECOMMERCE.SILVER.ORDERS B
#         ON A.ORDER_ID = B.ORDER_ID
#   WHERE
#     B.ORDER_STATUS = 'DELIVERED'
#   GROUP BY
#     1
# )
# SELECT
#   A.*,
#   B.UNIQUE_CUSTOMERS,
#   ROW_NUMBER() OVER (ORDER BY NET_REVENUE DESC) AS REVENUE_RANK,
#   ROW_NUMBER() OVER (PARTITION BY CATEGORY ORDER BY NET_REVENUE DESC) AS CATEGORY_RANK
# FROM
#   PRODUCT_KPIS A
#     INNER JOIN CUSTOMER_KPIS B
#       ON A.PRODUCT_ID = B.PRODUCT_ID;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
products = spark.table("ECOMMERCE.SILVER.PRODUCTS")
order_items = spark.table("ECOMMERCE.SILVER.ORDER_ITEMS")
orders = spark.table("ECOMMERCE.SILVER.ORDERS")


In [0]:
product_kpis = (
    products.alias("A")
    .join(
        order_items.alias("B"), F.col("A.PRODUCT_ID") == F.col("B.PRODUCT_ID"), "inner"
    )
    .groupBy(F.col("A.PRODUCT_ID"), F.col("A.PRODUCT_NAME"), F.col("A.CATEGORY"))
    .agg(
        F.sum("B.QUANTITY").alias("UNITS_SOLD"),
        F.countDistinct("B.ORDER_ID").alias("TOTAL_ORDERS"),
        F.sum("B.GROSS_AMOUNT").alias("GROSS_REVENUE"),
        F.sum("B.DISCOUNT_AMOUNT").alias("DISCOUNT_AMOUNT"),
        F.sum("B.NET_AMOUNT").alias("NET_REVENUE"),
    )
    .withColumn(
        "AVERAGE_SELLING_PRICE",
        F.when(F.col("UNITS_SOLD") != 0, F.col("NET_REVENUE") / F.col("UNITS_SOLD")),
    )
)

In [0]:
customer_kpis = (
    order_items.alias("A")
    .join(orders.alias("B"), F.col("A.ORDER_ID") == F.col("B.ORDER_ID"), "inner")
    .filter(F.col("B.ORDER_STATUS") == "DELIVERED")
    .groupBy(F.col("A.PRODUCT_ID"))
    .agg(F.countDistinct("B.CUSTOMER_ID").alias("UNIQUE_CUSTOMERS"))
)

In [0]:
overall_revenue_window = Window.orderBy(F.col("NET_REVENUE").desc())

category_revenue_window = Window.partitionBy("CATEGORY").orderBy(
    F.col("NET_REVENUE").desc()
)

In [0]:
final_product_kpis = (
    product_kpis.alias("A")
    .join(
        customer_kpis.alias("B"),
        F.col("A.PRODUCT_ID") == F.col("B.PRODUCT_ID"),
        "inner",
    )
    .select(
        F.col("A.*"),
        F.col("B.UNIQUE_CUSTOMERS"),
        F.row_number().over(overall_revenue_window).alias("REVENUE_RANK"),
        F.row_number().over(category_revenue_window).alias("CATEGORY_RANK"),
    )
)

In [0]:
display(final_product_kpis)

In [0]:
final_product_kpis.write.format("delta").mode("APPEND").saveAsTable(
    "ecommerce.gold.product_performance"
)

In [0]:
top_3_products = final_product_kpis.filter(F.col("CATEGORY_RANK") <= 3).orderBy(
    F.col("CATEGORY"), F.col("CATEGORY_RANK")
)

display(top_3_products)